# QA Generator - NL to Cypher Dataset

Generate pasangan (pertanyaan NL, Cypher query) untuk fine-tuning query model.

**Mengikuti pola dari `qa_generator.ipynb` reference.**

**Requirements:**
1. Python >= 3.10
2. Neo4j running di `bolt://localhost:7687`
3. Google Sheets API credentials di `.env`
4. Gemini API key di `.env`

**How to get Google API key:**
1. Go to https://console.cloud.google.com/
2. Enable Google Sheets API
3. Create service account credentials
4. Copy `GOOGLE_SHEETS_CLIENT_EMAIL` and `GOOGLE_SHEETS_PRIVATE_KEY` to `.env`

## Data Preparation

In [1]:
# Google Authentication & Config
import os, sys, json
import pandas as pd
from typing import Dict, List
from dotenv import load_dotenv

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

load_dotenv(os.path.join(ROOT_DIR, '.env'), override=True)

# --- Config ---
# TODO: Replace with your actual Google Spreadsheet ID
GOOGLE_SPREADSHEET_ID: str = os.getenv('GOOGLE_SPREADSHEET_ID', 'YOUR_SPREADSHEET_ID_HERE')
GOOGLE_SPREADSHEET_URL: str = f"https://docs.google.com/spreadsheets/d/{GOOGLE_SPREADSHEET_ID}/edit?usp=sharing"

GEMINI_API_KEY: str = os.getenv('GEMINI_API_KEY', '')
GOOGLE_SHEETS_CLIENT_EMAIL: str = os.getenv('GOOGLE_SHEETS_CLIENT_EMAIL', '')
GOOGLE_SHEETS_PRIVATE_KEY: str = os.getenv('GOOGLE_SHEETS_PRIVATE_KEY', '')

print(f"Root: {ROOT_DIR}")
print(f"Spreadsheet: {GOOGLE_SPREADSHEET_ID[:20]}...")
print(f"Gemini API: {'set' if GEMINI_API_KEY else 'NOT SET'}")
print(f"GSheets Email: {GOOGLE_SHEETS_CLIENT_EMAIL or 'NOT SET'}")

Root: d:\TA\llm-driven-legal-kg-visualization
Spreadsheet: 1oN5kMN_OI8WyITAQgJ3...
Gemini API: set
GSheets Email: sheets-access@thesis-llm-kg-research.iam.gserviceaccount.com


In [2]:
# Connect to Neo4j
from neo4j import GraphDatabase

NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'passwd123')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

with driver.session() as session:
    stats = session.run("""
        MATCH (n) 
        RETURN labels(n)[0] AS type, count(n) AS cnt 
        ORDER BY cnt DESC
    """).data()

print("Neo4j KG stats:")
for s in stats:
    print(f"  {s['type']}: {s['cnt']}")

Neo4j KG stats:
  Entity: 435


### Load seed data from KG

In [3]:
# Gather entity examples from Neo4j for template filling
entity_queries = {
    "Pasal": "MATCH (p:Pasal) RETURN p.label AS label, substring(p.content, 0, 200) AS content LIMIT 15",
    "Bab": "MATCH (b:Bab) RETURN b.label AS label, b.content AS content LIMIT 10",
    "EntitasHukum": "MATCH (e:EntitasHukum) RETURN e.label AS label, e.content AS content LIMIT 15",
    "PerbuatanHukum": "MATCH (ph:PerbuatanHukum) RETURN ph.label AS label, ph.content AS content LIMIT 15",
    "Sanksi": "MATCH (s:Sanksi) RETURN s.label AS label, s.content AS content LIMIT 15",
    "KonsepHukum": "MATCH (k:KonsepHukum) RETURN k.label AS label, k.content AS content LIMIT 15",
}

entity_data = {}
with driver.session() as session:
    for node_type, query in entity_queries.items():
        results = session.run(query).data()
        entity_data[node_type] = results
        print(f"{node_type}: {len(results)} entities loaded")

# Preview
for ntype, entities in entity_data.items():
    if entities:
        print(f"\n{ntype} examples:")
        for e in entities[:3]:
            print(f"  - {e['label']}")

Pasal: 15 entities loaded
Bab: 10 entities loaded
EntitasHukum: 15 entities loaded
PerbuatanHukum: 15 entities loaded
Sanksi: 15 entities loaded
KonsepHukum: 15 entities loaded

Pasal examples:
  - Pasal 5 ayat (1)
  - Pasal 20
  - Pasal 1

Bab examples:
  - BAB I
  - BAB II
  - BAB III

EntitasHukum examples:
  - penyelenggara negara
  - Orang
  - Badan Usaha

PerbuatanHukum examples:
  - melakukan perbuatan hukum
  - Pemanfaatan Teknologi Informasi dan Transaksi Elektronik
  - penggunaan Sistem Elektronik

Sanksi examples:
  - bertanggung jawab atas segala kerugian dan konsekuensi hukum yang timbul
  - pidana penjara paling lama 6 (enam) tahun dan/atau denda paling banyak Rp1.000.000.000,00
  - pidana penjara paling lama 12 (dua belas) tahun dan/atau denda paling banyak Rp2.000.000.000,00

KonsepHukum examples:
  - Informasi Elektronik
  - Transaksi Elektronik
  - Teknologi Informasi


### Step 1: Template-based generation

In [4]:
from finetuning.query_model.generate_training_data import (
    generate_from_templates, validate_cypher_queries, KG_SCHEMA,
    SYSTEM_INSTRUCTION, TrainingSample
)

template_samples = generate_from_templates(driver)

# Category breakdown
categories = {}
for s in template_samples:
    categories[s.category] = categories.get(s.category, 0) + 1

print(f"Total template-based pairs: {len(template_samples)}")
for cat, cnt in sorted(categories.items()):
    print(f"  {cat}: {cnt}")

Total template-based pairs: 278
  agregasi: 17
  definisi: 51
  entitas: 51
  hierarki: 95
  relasi: 37
  sanksi: 27


In [5]:
# Preview samples per category
import random

for cat in sorted(categories.keys()):
    cat_samples = [s for s in template_samples if s.category == cat]
    sample = random.choice(cat_samples)
    print(f"[{cat}]")
    print(f"  Q: {sample.question}")
    print(f"  C: {sample.response[:120]}...")
    print()

[agregasi]
  Q: BAB X memiliki berapa pasal?
  C: MATCH (b {label: 'BAB X'})-[:MEMUAT|MEMILIKI_PASAL]->(p:Pasal) RETURN b.label AS bab, count(p) AS jumlah_pasal...

[definisi]
  Q: Pasal berapa yang mendefinisikan tentang Penanda Tangan?
  C: MATCH (p:Pasal)-[:MENDEFINISIKAN]->(k:KonsepHukum) WHERE toLower(k.label) CONTAINS toLower('Penanda Tangan') RETURN p.la...

[entitas]
  Q: Apa kewajiban Penyelenggara Sertifikasi Elektronik asing menurut UU ITE?
  C: MATCH (p:Pasal)-[:BERLAKU_UNTUK]->(e:EntitasHukum) WHERE toLower(e.label) CONTAINS toLower('Penyelenggara Sertifikasi El...

[hierarki]
  Q: Sebutkan isi dari Pasal 2!
  C: MATCH (p:Pasal {label: 'Pasal 2'}) RETURN p.label AS pasal, p.content AS isi...

[relasi]
  Q: Apa hubungan antara Pasal 52 ayat (4) dan Pasal 27?
  C: MATCH (p1:Pasal {label: 'Pasal 52 ayat (4)'})-[r]->(p2:Pasal {label: 'Pasal 27'}) RETURN p1.label AS dari, type(r) AS re...

[sanksi]
  Q: Sanksi apa saja yang diatur dalam Pasal 50?
  C: MATCH (p:Pasal {label: 'Pa

### Synthetics data prompt

Prompt template untuk LLM-assisted generation — bisa di-load dari Google Sheets atau hardcoded.

In [6]:
# Synthetics data prompt (hardcoded — can be loaded from Google Sheets later)
from finetuning.query_model.generate_training_data import (
    LLM_GENERATION_PROMPT_SYSTEM, LLM_GENERATION_PROMPT_USER
)

print("SYSTEM PROMPT (first 500 chars):")
print(LLM_GENERATION_PROMPT_SYSTEM[:500])
print("\n...")
print(f"\nTotal system prompt length: {len(LLM_GENERATION_PROMPT_SYSTEM)} chars")

SYSTEM PROMPT (first 500 chars):
<ROLE>
You are an assistant specialized in generating synthetic NL-to-Cypher datasets for Indonesian legal Knowledge Graphs.
Your outputs must be valid Cypher queries that work with the following schema.
</ROLE>

<KG_SCHEMA>
Node Types:
- UndangUndang (properties: label, content, uu_number, source_document_id)
- Bab (properties: label, content)
- Pasal (properties: label, content)
- Ayat (properties: label, content)
- EntitasHukum (properties: label, content) — subjek/objek: orang, badan, instit

...

Total system prompt length: 1280 chars


## Generate QA pairs (LLM-assisted)

In [7]:
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel('gemini-2.5-flash')

print(f"Model ready: gemini-2.5-flash")

Model ready: gemini-2.5-flash


c:\Users\daffarafi\miniconda3\envs\ta-skripsi\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\daffarafi\AppData\Local\Temp\ipykernel_49168\3538115143.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


### Sanity check

In [8]:
# Sanity check: generate 3 pairs
example_entities_str = json.dumps(
    {k: [e['label'] for e in v[:5]] for k, v in entity_data.items()},
    ensure_ascii=False, indent=2
)
existing_qs = "\n".join([f"- {s.question}" for s in template_samples[:10]])

user_prompt = LLM_GENERATION_PROMPT_USER.format(
    total_data=3,
    example_entities=example_entities_str,
    existing_questions=existing_qs,
)

response = model.generate_content(
    [LLM_GENERATION_PROMPT_SYSTEM, user_prompt],
    generation_config={'response_mime_type': 'application/json', 'temperature': 0.9}
)

result = json.loads(response.text)
print(json.dumps(result, indent=4, ensure_ascii=False))

{
    "pairs": [
        {
            "question": "Sanksi apa yang ditetapkan untuk perbuatan hukum 'penggunaan Sistem Elektronik'?",
            "cypher": "MATCH (ph:PerbuatanHukum {content: 'penggunaan Sistem Elektronik'})<-[:MENGATUR]-(p:Pasal)-[:MENETAPKAN_SANKSI]->(s:Sanksi) RETURN s.content",
            "category": "sanksi"
        },
        {
            "question": "Entitas hukum siapa saja yang diatur dalam Pasal 20?",
            "cypher": "MATCH (p:Pasal {label: 'Pasal 20'})-[:BERLAKU_UNTUK]->(eh:EntitasHukum) RETURN eh.label",
            "category": "entitas"
        },
        {
            "question": "Pasal manakah yang mendefinisikan 'Sistem Elektronik'?",
            "cypher": "MATCH (p:Pasal)-[:MENDEFINISIKAN]->(kh:KonsepHukum {label: 'Sistem Elektronik'}) RETURN p.label, p.content",
            "category": "definisi"
        }
    ]
}


### Batch generate

In [9]:
from finetuning.query_model.generate_training_data import generate_with_llm

NUM_LLM_SAMPLES = 50  # Adjust as needed

llm_samples = generate_with_llm(
    driver, template_samples, NUM_LLM_SAMPLES, GEMINI_API_KEY
)
print(f"\nLLM-assisted: {len(llm_samples)} valid pairs generated")

  Batch 1/3: 20/20 valid
  Batch 2/3: 20/20 valid
  Batch 3/3: 10/10 valid

LLM-assisted: 50 valid pairs generated


## Validate & Save

In [10]:
# Combine and validate
all_samples = template_samples + llm_samples

valid_samples, errors = validate_cypher_queries(all_samples, driver)
print(f"Total: {len(all_samples)} -> Valid: {len(valid_samples)}, Invalid: {len(errors)}")

if errors:
    print(f"\nFirst 3 errors:")
    for err in errors[:3]:
        print(f"  Q: {err['question'][:60]}")
        print(f"  E: {err['error'][:80]}")
        print()

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing relationship type is: MEMILIKI_PASAL)} {position: line: 1, column: 45, offset: 44} for query: "EXPLAIN MATCH (b {label: 'BAB I'})-[:MEMUAT|MEMILIKI_PASAL]->(p:Pasal) RETURN p.label AS pasal, p.content AS isi"
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that

Total: 328 -> Valid: 328, Invalid: 0


In [11]:
# Save to CSV (local fallback)
from finetuning.query_model.generate_training_data import (
    save_to_csv, save_prompt_template_csv
)

OUTPUT_DIR = os.path.join(ROOT_DIR, 'finetuning', 'query_model', 'data')

train_path, val_path = save_to_csv(valid_samples, OUTPUT_DIR)
prompt_path = save_prompt_template_csv(OUTPUT_DIR)

print(f"Train: {train_path}")
print(f"Val:   {val_path}")
print(f"Prompt: {prompt_path}")

Train: d:\TA\llm-driven-legal-kg-visualization\finetuning\query_model\data\training_data.csv
Val:   d:\TA\llm-driven-legal-kg-visualization\finetuning\query_model\data\validation_data.csv
Prompt: d:\TA\llm-driven-legal-kg-visualization\finetuning\query_model\data\prompt_data.csv


In [12]:
# Final stats
import csv

for name, path in [('Train', train_path), ('Val', val_path)]:
    with open(path, 'r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    cats = {}
    for r in rows:
        cats[r['category']] = cats.get(r['category'], 0) + 1
    print(f"{name}: {len(rows)} samples")
    for cat, cnt in sorted(cats.items()):
        print(f"  {cat}: {cnt}")
    print()

Train: 262 samples
  agregasi: 17
  definisi: 46
  entitas: 48
  hierarki: 74
  multi-hop: 7
  relasi: 39
  sanksi: 31

Val: 66 samples
  agregasi: 3
  definisi: 14
  entitas: 11
  hierarki: 22
  multi-hop: 1
  relasi: 10
  sanksi: 5



## (Optional) Upload to Google Sheets

In [20]:
# Upload to Google Sheets (skip if credentials not configured)
if GOOGLE_SHEETS_CLIENT_EMAIL and GOOGLE_SPREADSHEET_ID != 'YOUR_SPREADSHEET_ID_HERE':
    from finetuning.query_model.generate_training_data import upload_to_google_sheets
    upload_to_google_sheets(valid_samples, GOOGLE_SPREADSHEET_ID, "training_data", "validation_data", client_email=GOOGLE_SHEETS_CLIENT_EMAIL, private_key=GOOGLE_SHEETS_PRIVATE_KEY)
else:
    print("Google Sheets not configured. Data saved to CSV only.")
    print(f"To upload later, set GOOGLE_SPREADSHEET_ID and credentials in .env")

Uploading 262 train + 66 val samples to Sheets...

--- Writing 262 rows to 'training_data' ---


  0%|          | 0/27 [00:00<?, ?it/s]2026-04-05 10:53:42,515 - INFO - Successfully wrote row 1/262
2026-04-05 10:53:44,415 - INFO - Successfully wrote row 2/262
2026-04-05 10:53:45,937 - INFO - Successfully wrote row 3/262
2026-04-05 10:53:47,447 - INFO - Successfully wrote row 4/262
2026-04-05 10:53:48,997 - INFO - Successfully wrote row 5/262
2026-04-05 10:53:50,580 - INFO - Successfully wrote row 6/262
2026-04-05 10:53:52,212 - INFO - Successfully wrote row 7/262
2026-04-05 10:53:53,777 - INFO - Successfully wrote row 8/262
2026-04-05 10:53:56,064 - INFO - Successfully wrote row 9/262
2026-04-05 10:53:57,606 - INFO - Successfully wrote row 10/262
  4%|▎         | 1/27 [00:19<08:14, 19.02s/it]2026-04-05 10:54:01,215 - INFO - Successfully wrote row 11/262
2026-04-05 10:54:02,762 - INFO - Successfully wrote row 12/262
2026-04-05 10:54:04,321 - INFO - Successfully wrote row 13/262
2026-04-05 10:54:05,905 - INFO - Successfully wrote row 14/262
2026-04-05 10:54:07,560 - INFO - Successful

  ✓ 262 rows written successfully

--- Writing 66 rows to 'validation_data' ---


  0%|          | 0/7 [00:00<?, ?it/s]2026-04-05 11:06:50,752 - INFO - Successfully wrote row 1/66
2026-04-05 11:06:52,381 - INFO - Successfully wrote row 2/66
2026-04-05 11:06:53,958 - INFO - Successfully wrote row 3/66
2026-04-05 11:06:55,575 - INFO - Successfully wrote row 4/66
2026-04-05 11:06:57,148 - INFO - Successfully wrote row 5/66
2026-04-05 11:06:58,730 - INFO - Successfully wrote row 6/66
2026-04-05 11:07:00,346 - INFO - Successfully wrote row 7/66
2026-04-05 11:07:01,982 - INFO - Successfully wrote row 8/66
2026-04-05 11:07:03,566 - INFO - Successfully wrote row 9/66
2026-04-05 11:07:05,157 - INFO - Successfully wrote row 10/66
 14%|█▍        | 1/7 [00:17<01:47, 17.97s/it]2026-04-05 11:07:08,762 - INFO - Successfully wrote row 11/66
2026-04-05 11:07:10,357 - INFO - Successfully wrote row 12/66
2026-04-05 11:07:11,910 - INFO - Successfully wrote row 13/66
2026-04-05 11:07:13,492 - INFO - Successfully wrote row 14/66
2026-04-05 11:07:15,076 - INFO - Successfully wrote row 15/

  ✓ 66 rows written successfully

Google Sheets upload complete!


In [15]:
driver.close()
print("Done!")

Done!
